# 4.1 Aggregation changes the question

**Learning objectives:** convert WGS84 points to H3; aggregate continuous scores; preserve counts; identify scale effects.

**How to learn:** predict - run - explain - change one thing. Spend 45–90 minutes here.

**Data:** the default worked example is deterministic synthetic practice data near Colorado. It is not SNOTEL, a watershed survey, a burn map, or a foundation-model output.


## Before you run

H3 takes latitude then longitude. Aggregate scores, not arbitrary numeric class IDs. A mean at sample points is not an area-weighted fraction unless the sampling design justifies it. Sparse cells need their counts displayed.


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course import OUT, points


## Build a prediction surface, then bin it


In [2]:
from course import embeddings,evaluate,h3_summary
frame=points(2000)
model,*_=evaluate(frame,embeddings(frame))
frame['probability']=model.predict_proba(embeddings(frame))[:,1]
for resolution in [6,9]:
    summary=h3_summary(frame,resolution)
    assert summary['count'].sum()==len(frame)
    summary.to_csv(OUT/f'h3_r{resolution}.csv',index=False)
    weighted=np.average(summary.mean_probability,weights=summary['count'])
    np.testing.assert_allclose(weighted,frame.probability.mean())
    print(resolution,'cells:',len(summary),'median count:',summary['count'].median())


6 cells: 17 median count: 95.0
9 cells: 1509 median count: 1.0


## Your turn

Compute resolution 7 and filter out cells with fewer than five samples. Compare weighted and unweighted cell means.

Use a new cell below to try this before opening the solution.


In [3]:
# Write your experiment here. The worked example above is already complete.


<details><summary>Solution and reasoning - open after trying</summary>

Use `h3_summary(frame,7)` and `summary[summary["count"]>=5]`. An unweighted mean gives sparse and dense cells equal influence. Filtering alters the cohort: report how many points and cells were excluded.

</details>


## Check yourself

Expected: counts sum to 2,000 at both resolutions, and weighted means reproduce the point mean. Does resolution 9 imply parcel-level accuracy? No: support and model quality are independent of display resolution.

**Exit ticket:** write one result, one explanation, and one limitation. Save the notebook with your own observations.


## Transfer to real data

Replace the synthetic table with independently evaluated real predictions including longitude, latitude and probability. Preserve masks and sampling weights. H3 cells do not follow watershed boundaries.
